In [1]:
import os
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
from tqdm import tqdm
from torch import optim
import torchvision
import torchvision.transforms as trfm
from torch.utils.data import DataLoader
import torch.nn.functional as funcy

In [2]:
ep=250
batch=4
imsi=64
lr=3e-4

In [3]:
dev='cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
data=torchvision.datasets.ImageFolder('/kaggle/input',transform=trfm.Compose([trfm.Resize(80),trfm.RandomResizedCrop(imsi,scale=(0.8,1)),
                                                                                                trfm.ToTensor(),trfm.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))]))

In [5]:
data[1]

(tensor([[[-1.0000, -1.0000, -0.9922,  ..., -0.9922, -0.9922, -0.9922],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -0.9922, -0.9922],
          [-1.0000, -1.0000, -0.9922,  ..., -0.9922, -0.9922, -1.0000],
          ...,
          [-0.9843, -0.9843, -0.9922,  ..., -0.9843, -0.9765, -0.9843],
          [-0.9922, -0.9843, -0.9922,  ..., -0.9922, -0.9843, -0.9922],
          [-0.9922, -0.9922, -0.9922,  ..., -0.9843, -0.9922, -0.9922]],
 
         [[-0.6941, -0.6863, -0.6784,  ..., -0.6627, -0.6706, -0.6706],
          [-0.6549, -0.6549, -0.6471,  ..., -0.6235, -0.6314, -0.6314],
          [-0.6078, -0.6078, -0.6000,  ..., -0.5765, -0.5843, -0.5922],
          ...,
          [-0.7412, -0.7412, -0.7647,  ..., -0.9216, -0.9216, -0.9294],
          [-0.7882, -0.8118, -0.8275,  ..., -0.9373, -0.9294, -0.9451],
          [-0.8353, -0.8510, -0.8431,  ..., -0.9451, -0.9451, -0.9451]],
 
         [[-0.6549, -0.6549, -0.6471,  ..., -0.6235, -0.6314, -0.6392],
          [-0.6157, -0.6157,

In [6]:
loader=DataLoader(data,batch_size=batch,shuffle=True)

In [7]:
class Att(nn.Module):
    def __init__(self,chl,size):
        super().__init__()
        self.chl=chl
        self.si=size
        self.mha=nn.MultiheadAttention(chl,4,batch_first=True)
        self.lm=nn.LayerNorm([chl])
        self.lin=nn.Sequential(nn.LayerNorm([chl]),nn.Linear(chl,chl),nn.GELU(),nn.Linear(chl,chl))
    def forward(self,x):
        x=x.view(-1,self.chl,self.si*self.si).swapaxes(1,2)
        ln=self.lm(x)
        atv,_=self.mha(ln,ln,ln)
        atv=atv+x
        atv=self.lin(atv)+atv
        return atv.swapaxes(2,1).view(-1,self.chl,self.si,self.si)

In [8]:
class DDD(nn.Module):
    def __init__(self,inn,out,mchl=None,resi=False):
        super().__init__()
        self.res=resi
        if not mchl:
            mchl=out
        self.dconv=nn.Sequential(nn.Conv2d(inn,mchl,kernel_size=3,padding=1,bias=False),
                                nn.GroupNorm(1,mchl),nn.GELU(),
                                nn.Conv2d(mchl,out,kernel_size=3,padding=1,bias=False),
                                nn.GroupNorm(1,out))
    def forward(self,x):
        return funcy.gelu(x+self.dconv(x)) if self.res else self.dconv(x)

In [9]:
class Down(nn.Module):
    def __init__(self,inn,out,emd=256):
        super().__init__()
        self.conv=nn.Sequential(nn.MaxPool2d(2),DDD(inn,inn,resi=True),DDD(inn,out))
        self.embl=nn.Sequential(nn.SiLU(),nn.Linear(emd,out))
    def forward(self,x,t):
        x=self.conv(x)
        embb=self.embl(t)[:,:,None,None].repeat(1,1,x.shape[-2],x.shape[-1])
        return x+embb

In [10]:
class Up(nn.Module):
    def __init__(self,inn,out,emd=256):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True)
        self.conv=nn.Sequential(DDD(inn,inn,resi=True),DDD(inn,out,inn//2))
        self.embl=nn.Sequential(nn.SiLU(),nn.Linear(emd,out))
    def forward(self,x,sx,t):
        x=self.up(x)
        x=torch.cat([sx,x],dim=1)
        x=self.conv(x)
        embb=self.embl(t)[:,:,None,None].repeat(1,1,x.shape[-2],x.shape[-1])
        return x+embb

In [11]:
class UUU(nn.Module):
    def __init__(self,inn=3,out=3,dim=256,device=dev):
        super().__init__()
        self.dev=device
        self.dim=dim
        self.inc=DDD(inn,64)
        self.d1=Down(64,128)
        #self.a1=Att(128,32)
        self.d2=Down(128,256)
        self.a2=Att(256,16)
        self.d3=Down(256,256)
        self.a3=Att(256,8)

        self.b1=DDD(256,512)
        self.b2=DDD(512,512)
        self.b3=DDD(512,256)

        self.u1=Up(512,128)
        self.a4=Att(128,16)
        self.u2=Up(256,64)
        #self.a5=Att(64,32)
        self.u3=Up(128,64)
        #self.a6=Att(64,64)
        self.out=nn.Conv2d(64,out,kernel_size=1)
    def encod(self,t,chl):
        invf=1.0/(10000**(torch.arange(0,chl,2,device=self.dev)).float()/chl)
        sin=torch.sin(t.repeat(1,chl//2)*invf)
        cos=torch.cos(t.repeat(1,chl//2)*invf)
        enc=torch.cat([sin,cos],dim=-1)
        return enc
    def forward(self,x,t):
        t=t.unsqueeze(-1).type(torch.float)
        t=self.encod(t,self.dim)
        x1 = self.inc(x)
        x2 = self.d1(x1, t)
        #x2 = self.a1(x2)
        x3 = self.d2(x2, t)
        x3 = self.a2(x3)
        x4 = self.d3(x3, t)
        x4 = self.a3(x4)

        x4 = self.b1(x4)
        x4 = self.b2(x4)
        x4 = self.b3(x4)

        x = self.u1(x4, x3, t)
        x = self.a4(x)
        x = self.u2(x, x2, t)
        #x = self.a5(x)
        x = self.u3(x, x1, t)
        #x = self.a6(x)
        output = self.out(x)
        return output

In [12]:
model=UUU().to(dev)

In [13]:
class Difuse:
    def __init__(self,noi=1000,sb=1e-4,eb=0.02,imgsi=256,device=dev):
        self.noi=noi
        self.sb=sb
        self.eb=eb
        self.imsi=imgsi
        self.dev=device
        self.beta=self.noisch().to(device)
        self.alpha=1.-self.beta
        self.ialpha=torch.cumprod(self.alpha,dim=0)
    def noisch(self):
        return torch.linspace(self.sb,self.eb,self.noi)
    def noimg(self,x,t):
        sqialpha=torch.sqrt(self.ialpha[t])[:,None,None,None]
        sq1ialpha=torch.sqrt(1-self.ialpha[t])[:,None,None,None]
        e=torch.randn_like(x)
        return sqialpha*x+sq1ialpha*e,e
    def sample_timesteps(self, n):
        return torch.randint(low=1, high=self.noi, size=(n,))
    def sample(self, model, n):
        model.eval()
        with torch.no_grad():
            x = torch.randn((n, 3, self.imsi, self.imsi)).to(self.dev)
            for i in tqdm(reversed(range(1, self.noi)),position=0):
                t = (torch.ones(n) * i).long().to(self.dev)
                predicted_noise = model(x, t)
                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.ialpha[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]
                if i > 1:
                    noise = torch.randn_like(x)
                else:
                    noise = torch.zeros_like(x)
                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / (torch.sqrt(1 - alpha_hat))) * predicted_noise) + torch.sqrt(beta) * noise
        model.train()
        x = (x.clamp(-1, 1) + 1) / 2
        x = (x * 255).type(torch.uint8)
        return x

In [14]:
def save_images(images, path, **kwargs):
    grid = torchvision.utils.make_grid(images, **kwargs)
    ndarr = grid.permute(1, 2, 0).to('cpu').numpy()
    im = Image.fromarray(ndarr)
    im.save(path)

In [15]:
from PIL import Image

In [16]:
os.makedirs("/kaggle/working/results/ddpm", exist_ok=True)
os.makedirs("/kaggle/working/models/ddpm", exist_ok=True)

In [17]:
optimizer = optim.AdamW(model.parameters(), lr=lr)
mse = nn.MSELoss()
diffusion = Difuse(imgsi=64, device=dev)
l = len(loader)
for epoch in range(ep):
    pbar = tqdm(loader)
    for i, (images, _) in enumerate(pbar):
        images = images.to(dev)
        t = diffusion.sample_timesteps(images.shape[0]).to(dev)
        x_t, noise = diffusion.noimg(images, t)
        predicted_noise = model(x_t, t)
        loss = mse(noise, predicted_noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pbar.set_postfix(MSE=loss.item())

    sampled_images = diffusion.sample(model, n=images.shape[0])
    save_images(sampled_images, os.path.join("/kaggle/working/results",'ddpm', f"{epoch}.jpg"))
    torch.save(model.state_dict(), os.path.join("/kaggle/working/models", 'ddpm', f"ckpt.pt"))

100%|██████████| 1080/1080 [01:52<00:00,  9.58it/s, MSE=0.105]
999it [00:14, 69.72it/s]
100%|██████████| 1080/1080 [01:39<00:00, 10.86it/s, MSE=0.225]
999it [00:14, 69.09it/s]
100%|██████████| 1080/1080 [01:38<00:00, 11.02it/s, MSE=0.0188]
999it [00:14, 69.40it/s]
100%|██████████| 1080/1080 [01:38<00:00, 11.00it/s, MSE=0.0183]
999it [00:14, 69.55it/s]
100%|██████████| 1080/1080 [01:38<00:00, 11.00it/s, MSE=0.00699]
999it [00:14, 69.52it/s]
100%|██████████| 1080/1080 [01:37<00:00, 11.05it/s, MSE=0.0848]
999it [00:14, 69.40it/s]
100%|██████████| 1080/1080 [01:37<00:00, 11.11it/s, MSE=0.00628]
999it [00:14, 69.43it/s]
100%|██████████| 1080/1080 [01:38<00:00, 11.01it/s, MSE=0.0191]
999it [00:14, 69.29it/s]
100%|██████████| 1080/1080 [01:37<00:00, 11.09it/s, MSE=0.0157]
999it [00:14, 69.33it/s]
100%|██████████| 1080/1080 [01:40<00:00, 10.77it/s, MSE=0.0304]
999it [00:14, 67.22it/s]
100%|██████████| 1080/1080 [01:42<00:00, 10.57it/s, MSE=0.0125]
999it [00:14, 67.98it/s]
100%|██████████| 1080